# EDA on log2FC values from current bulk RNA-seq dataset

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
root = Path.cwd().parent
sys.path.insert(0, str(root))

Data root configs.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    read_l2fc_as_df,
    bind_l2fc_data,
    get_l2fc_and_cfu_data,
    attach_synergy_metadata,
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)
# Load annotations
annotations = pd.read_table(annot_path, sep = "\t")
annotations.set_index("TIGR4.old", inplace = True, drop = True)

# Get log2fc data
l2fc_df = attach_synergy_metadata(get_l2fc_and_cfu_data(l2fc_dir, cfu_dir, time_matched = True))

# Get adjusted pvalues for DEGs
_, pval_df_list, ids = read_l2fc_as_df(
    data_dir = l2fc_dir,
    time_matched = True,
    pval = True
)
pval_df = bind_l2fc_data(
    l2fc_df_list = pval_df_list,
    ids = ids
)
pval_df = pval_df.iloc[~pval_df.index.str.contains("NDC")]

# Bliss score and simple interaction score
synergy_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
synergy_df = synergy_df.dropna(axis = 1)

## PCA on log2FC data

Run PCA and color by drug.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Sample list
drug_list = ["CEF", "CIP", "CEF+CIP"]

# Filter to genes
X = l2fc_df[l2fc_df["drug_id"].isin(drug_list)]
X = X.iloc[:, X.columns.str.contains("SP")]
X = X.dropna(axis = 1)

# Run PCA
pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components = 2))
])
pca_results = pd.DataFrame(pca.fit_transform(X), index = X.index)
pca_results.columns = ["PC1", "PC2"]
pca_results = pd.merge(pca_results, l2fc_df, left_index = True, right_index = True, how = "left")

# Plot
sns.scatterplot(data = pca_results, x = "PC1", y = "PC2", hue = "drug_id", style = "drug1_dose")

## DEGs over time and dose

In [ ]:
def get_deg_count(
    l2fc_df,
    pval_df,
    pval_cutoff,
    l2fc_cutoff
):
    # Identify gene columns
    gene_cols = l2fc_df.columns[
        l2fc_df.columns.str.contains("SP")
    ]

    # Separate metadata and gene-level values
    meta = l2fc_df.drop(columns = gene_cols)
    l2fc = l2fc_df.loc[:, gene_cols]

    # Align p-values with both the rows and columns of l2fc
    pval = pval_df.reindex(
        index = l2fc.index,
        columns = gene_cols
    )

    # A gene is a DEG when it passes both thresholds
    deg_mask = (
        l2fc.abs().gt(l2fc_cutoff)
        & pval.lt(pval_cutoff)
    )

    # Count DEGs per sample
    out = meta.copy()
    out.insert(0, "num_deg", deg_mask.sum(axis = 1))

    return out

In [ ]:
df = get_deg_count(
    l2fc_df = l2fc_df,
    pval_df = pval_df,
    pval_cutoff = 0.05,
    l2fc_cutoff = 1
)

# Drug of interst
drug = "RIF"

sns.lineplot(
    data = df[df["drug_id"] == drug],
    x = "timepoint",
    y = "num_deg",
    hue = "drug1_dose"
)
plt.xlabel("Time (h)")
plt.ylabel("Number of DEGs")
plt.title(f"Number of DEGs over time for {drug}")

## Heatmaps for log2FC data

Annotation categories.

In [ ]:
print(annotations["Category1"].unique())

Plot specific pathways and samples in heatmap.

In [ ]:
from src.eda import plot_l2fc_heatmap, find_consistent_interaction_genes

drug_order = ["CIP", "CEF", "VNC", "RIF"]
annot_col = "Category1"
gene_categories = ["Lipid metabolism"]

plot_l2fc_heatmap(
    df = l2fc_df,
    annotations = annotations,
    secondary_annot_col = "Product",
    drug_order = drug_order,
    annot_col = annot_col,
    gene_categories = gene_categories,
    figsize = (19, 9),
    show_xticklabels = True,
    show_yticklabels = False,
    vmax = 2.5
)

Examine single-drug DEGs.

In [ ]:
import numpy as np

drug = "CEF+RIF"
single_df = l2fc_df[l2fc_df["drug_id"] == drug]
single_df = single_df.sort_values(["timepoint", "drug1_dose"])

# Filter to 
gene_cols = l2fc_df.columns[l2fc_df.columns.str.contains("^SP", regex = True, na = False)]
samples = l2fc_df.index[l2fc_df["drug_id"] == drug]
filtered_l2fc = l2fc_df.loc[samples][gene_cols]
filtered_pval = pval_df.loc[samples]

# Filter to fdr < 0.05
bool_pval = (filtered_pval < 0.05) & (~filtered_pval.isna())
bool_l2fc = abs(filtered_l2fc) > 1

# 
min_fraction = 0.5
l2fc_cutoff = 1
padj_cutoff = 0.05

sig_pval = filtered_pval < padj_cutoff

up_deg = (filtered_l2fc > l2fc_cutoff) & sig_pval
down_deg = (filtered_l2fc < -l2fc_cutoff) & sig_pval

up_fraction = up_deg.mean(axis = 0)
down_fraction = down_deg.mean(axis = 0)

consistent_deg = pd.DataFrame({
    "drug": drug,
    "n_samples": filtered_l2fc.notna().sum(axis = 0),
    "up_count": up_deg.sum(axis = 0),
    "up_fraction": up_fraction,
    "down_count": down_deg.sum(axis = 0),
    "down_fraction": down_fraction,
    "mean_l2fc": filtered_l2fc.mean(axis = 0),
    "median_l2fc": filtered_l2fc.median(axis = 0),
})

consistent_deg["direction"] = np.select(
    [
        consistent_deg["up_fraction"] >= min_fraction,
        consistent_deg["down_fraction"] >= min_fraction,
    ],
    [
        "up",
        "down",
    ],
    default = "none",
)

consistent_deg = consistent_deg[consistent_deg["direction"] != "none"]
consistent_deg["max_fraction"] = consistent_deg[["up_fraction", "down_fraction"]].max(axis = 1)

consistent_deg = consistent_deg.sort_values(
    ["direction", "max_fraction", "mean_l2fc"],
    ascending = [False, False, False]
)


# Heatmap 
fig, ax = plt.subplots(figsize = (13, 50))
sns.heatmap(
    data = single_df[consistent_deg.index].T, 
    cmap = "coolwarm", 
    vmax = 5,
    vmin = -5,
    ax = ax,
    cbar_kws = {"label": "Log2FC"}
)

# Add annotations as extra axis
annots = annotations.reindex(consistent_deg.index)
secax = ax.secondary_yaxis("left")
secax.set_yticks(np.arange(len(annots)) + 0.5)
secax.set_yticklabels(annots["Product"])
secax.spines["left"].set_position(("outward", 100))
secax.set_ylabel("Product")

ax.set_title(f"Top consistently high/low DEGs for {drug}")

## Transcriptional interaction scores

Distribution of interaction scores.

In [ ]:
# Subset to combo I'm interested in
combo = "CIP+VNC"
combo_df = synergy_df[synergy_df["drug_id"] == combo]

# Get interation scores
interaction_scores = combo_df.iloc[:, combo_df.columns.str.contains("SP")].values.ravel()

# Plot histogram
fig, ax = plt.subplots()
ax.hist(interaction_scores, bins = 40)
ax.set_title(f"Distribution of interaction scores for {combo}")
ax.set_xlabel("Transcriptional interaction score")
ax.set_ylabel("Frequency")
plt.show()

Set interaction score significance cutoff, then use the  

In [ ]:
import numpy as np
from src.eda import find_consistent_interaction_genes

# Top x percent of interaction scores to use
top_percent = 0.05

# Get 95 and 5 quantiles
left_cutoff = np.quantile(interaction_scores, top_percent)
right_cutoff = np.quantile(interaction_scores, 1- top_percent)

print(f"{1 - top_percent} quantile : {right_cutoff}")
print(f"{top_percent} quantile : {left_cutoff}" )

# Find genes that are either consistenly above or below cutoffs in 50% of samples
consistent_interaction_genes = find_consistent_interaction_genes(
    df = synergy_df,
    combo = combo,
    left_cutoff = left_cutoff,
    right_cutoff = right_cutoff,
    min_fraction = 0.50,
)

# Heatmap 
fig, ax = plt.subplots(figsize = (15, 9))
sns.heatmap(
    data = combo_df[consistent_interaction_genes.index].T, 
    cmap = "coolwarm", 
    ax = ax, 
    vmax = 7,
    vmin = -7,
    cbar_kws = {"label": "Transcriptional interaction score"}
)

# Add annotations as extra axis
annots = annotations.reindex(consistent_interaction_genes.index)
secax = ax.secondary_yaxis("left")
secax.set_yticks(np.arange(len(annots)) + 0.5)
secax.set_yticklabels(annots["Product"])
secax.spines["left"].set_position(("outward", 80))
secax.set_ylabel("Product")

ax.set_title(f"Top consistently high/low interaction scores for {combo}")

In [ ]:
drug_order = ["CIP", "VNC", "CIP+VNC"]
annot_col = "Category1"
genes = consistent_interaction_genes.index
plot_l2fc_heatmap(
    df = l2fc_df,
    annotations = annotations,
    drug_order = drug_order,
    annot_col = annot_col,
    figsize = (15, 9),
    show_xticklabels = True,
    show_yticklabels = True,
    genes = genes,
    vmax = 5
)

## Synergy scores

In [ ]:
sns.stripplot(synergy_df, x = "drug_id", y = "synergy_score", hue = "timepoint")
plt.title("Synergy score distribution")
plt.xlabel("Drug combination")
plt.ylabel("EOB synergy score")